# Design of Experiments (DOE)

This notebook demonstates use of Design Of Experiment (DOE) capabilities included in the ProcessOptimizer code.

First, we define the search space that we are interest in like we would for a normal optimization problem.
We do this using the Space class from the ProcessOptimizer library.

Note that we have also included a 2-level categorical variable.
At current, the version of D-optimal designs in ProcessOptimizer does not support categorical variables with more than two levels.

In [1]:
import numpy as np

from ProcessOptimizer.space import Categorical, Integer, Real, Space

factor_space = Space(dimensions=[
    Real(10, 40, name='var_x1'),
    Integer(20, 100, name='var_x2'),
    Integer(-30, 30, name='var_x3'),
    Categorical(['a', 'b'], name='var_x4')
    ])

## D-optimal design

D-optimal design is a type of experimental design that is used to find the optimal threatment combinations for a given number of experiments.

Besides the search space, we also need to define the number of experiments that we want to run and the type of model we want to be able to fit to the data.

In [2]:
from ProcessOptimizer.doe import get_optimal_DOE

number_of_experiments = 12

design, factor_names = get_optimal_DOE(factor_space, number_of_experiments)

print("Factor names:")
print(factor_names)
print("Design:")
print(design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[[np.float64(10.0) np.int64(20) np.int64(30) 'b']
 [np.float64(40.0) np.int64(20) np.int64(30) 'a']
 [np.float64(10.0) np.int64(100) np.int64(30) 'a']
 [np.float64(10.0) np.int64(20) np.int64(30) 'a']
 [np.float64(40.0) np.int64(20) np.int64(-30) 'b']
 [np.float64(40.0) np.int64(20) np.int64(-30) 'a']
 [np.float64(10.0) np.int64(20) np.int64(-30) 'b']
 [np.float64(10.0) np.int64(100) np.int64(-30) 'b']
 [np.float64(40.0) np.int64(100) np.int64(30) 'b']
 [np.float64(40.0) np.int64(20) np.int64(30) 'b']
 [np.float64(40.0) np.int64(100) np.int64(-30) 'a']
 [np.float64(10.0) np.int64(20) np.int64(-30) 'a']]


This is consistent with the type of each dimension in the search space, but not particularly print friendly. Thus we convert the numbers to strings and get:

In [3]:
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['10.0' '20' '30' 'b']
 ['40.0' '20' '30' 'a']
 ['10.0' '100' '30' 'a']
 ['10.0' '20' '30' 'a']
 ['40.0' '20' '-30' 'b']
 ['40.0' '20' '-30' 'a']
 ['10.0' '20' '-30' 'b']
 ['10.0' '100' '-30' 'b']
 ['40.0' '100' '30' 'b']
 ['40.0' '20' '30' 'b']
 ['40.0' '100' '-30' 'a']
 ['10.0' '20' '-30' 'a']]


Besides the design space and number of experiments, a number of other parameters can be set. These are:
- `design_type` : Various keywords can be used to specify the regression model that the design should be optimal for.
- `model` : As an alternative to `design_type`, a model can be specified directly. `design_type` and `model` are mutually exclusive. If both are specified, `design_type` will be used.
- `replicates` : A number of replicates can be specified. This is useful when the same experiment is to be run multiple times.
- `sorting` :  Various keywords can be used to specify sorting of experiments in the design.
- `res` : A resolution can be specified. This controls how coarsely the factors should be sampled during the design optimization. 

## design_type

`design_type` can be set to one of the following:
- `linear` : Simple linear regression model without interaction terms.
- `screening` : Screening design with main effects and two-factor interactions.This will be the default if no `design_type` or `model` is specified.
- `response` : Response surface design with main effects, two-factor interactions, and quadratic effects for non-categorical factors.
- `optimization` : Optimization design with main effects, two- and three-factor interactions, and quadratic and cubic effects for non-categorical factors.

Lets see the difference when we use a different one than the screening design, which we used before, as it is the default.

If we keep the experimental budget the same, we see that we get an error. We cannot fit a model with 14 parameters using only 12 experiments.

In [4]:
number_of_experiments = 12

design, factor_names = get_optimal_DOE(factor_space, number_of_experiments, design_type='response')
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

ValueError: Can't build a design of size 12 for a model of rank 14. Model: '(var_x1+var_x2+var_x3+var_x4)**2+pow(var_x1, 2)+pow(var_x2, 2)+pow(var_x3, 2)'

Let us try increase the budget a bit.

In [32]:
number_of_experiments = 15

design, factor_names = get_optimal_DOE(factor_space, number_of_experiments, design_type='response')
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['28.000000000000004' '20' '-6' 'a']
 ['25.0' '20' '30' 'b']
 ['40.0' '20' '-30' 'b']
 ['34.0' '100' '-30' 'b']
 ['10.0' '60' '-30' 'b']
 ['40.0' '20' '30' 'a']
 ['40.0' '100' '30' 'a']
 ['40.0' '68' '12' 'b']
 ['10.0' '20' '-30' 'a']
 ['10.0' '20' '0' 'b']
 ['10.0' '100' '30' 'b']
 ['10.0' '100' '-30' 'a']
 ['40.0' '68' '-30' 'a']
 ['10.0' '20' '30' 'a']
 ['19.0' '76' '30' 'a']]


Note, that in the `screening` design we primarily sample the corners of the search space, while in the `response` design we also sample values of the numerical dimensions, that are near the center of the specified range to fit the quadratic effects.

## res

Lets next change the `res` parameter to see how that affects the design. Default resolution is 11. We generally want to use odd rather than even numbers for resolution. This ensures that we sample the center point of the numerical dimensions. 

In [33]:
number_of_experiments = 15
resolution = 3

design, factor_names = get_optimal_DOE(factor_space,
                                       number_of_experiments,
                                       design_type='response',
                                       res=resolution)
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['40.0' '100' '-30' 'b']
 ['25.0' '20' '-30' 'b']
 ['10.0' '100' '30' 'a']
 ['25.0' '100' '30' 'b']
 ['10.0' '20' '30' 'b']
 ['25.0' '60' '0' 'a']
 ['10.0' '20' '-30' 'a']
 ['10.0' '100' '0' 'b']
 ['40.0' '100' '30' 'a']
 ['10.0' '60' '-30' 'b']
 ['40.0' '20' '-30' 'a']
 ['10.0' '100' '-30' 'a']
 ['40.0' '60' '30' 'b']
 ['40.0' '20' '30' 'a']
 ['40.0' '20' '0' 'b']]


With a resolution of 3, we see that that only values for each dimension present in the design are the minimum, maximum and center values.

In [35]:
number_of_experiments = 15
resolution = 31

design, factor_names = get_optimal_DOE(factor_space,
                                       number_of_experiments,
                                       design_type='response',
                                       res=resolution)
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['40.0' '20' '30' 'a']
 ['10.0' '63' '30' 'a']
 ['23.0' '20' '-30' 'b']
 ['40.0' '20' '4' 'b']
 ['10.0' '20' '-30' 'a']
 ['40.0' '100' '30' 'a']
 ['40.0' '60' '-30' 'a']
 ['40.0' '20' '-30' 'b']
 ['10.0' '20' '30' 'b']
 ['40.0' '100' '-30' 'b']
 ['10.0' '100' '30' 'b']
 ['25.0' '20' '0' 'a']
 ['34.0' '63' '30' 'b']
 ['10.0' '63' '-18' 'b']
 ['10.0' '100' '-30' 'a']]


With a resolution of 31, we see that we now have all integer values between the minimum and maximum values for `var_x1` possible. For `var_x3`, where the total range is 60, we have all even values between -30 and 30 possible.

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['10.0' '100' '-30' 'b']
 ['40.0' '20' '30' 'a']
 ['10.0' '20' '30' 'a']
 ['10.0' '20' '-30' 'b']
 ['40.0' '20' '-30' 'a']
 ['40.0' '100' '-30' 'a']
 ['10.0' '100' '30' 'a']
 ['40.0' '68' '3' 'a']
 ['24.5' '100' '-1' 'b']
 ['10.0' '61' '30' 'b']
 ['40.0' '20' '30' 'b']
 ['40.0' '100' '30' 'b']
 ['10.0' '60' '-30' 'a']
 ['40.0' '61' '-30' 'b']
 ['26.5' '68' '30' 'a']]


We generally get more optimal designs with better resolution but also a higher number of different values for the numerical dimensions in the design. There can thus be a trade-off between how complicated the experiment will be to do in practice and the quality of the design.

In [99]:
points_array = np.array([[0.2352, -1, 0.34212, -0.4523, 1, -0.7],
                [0.253525, -1, 0., -0.41234, 1, 0.7]
                ])

res = 6

bins = np.linspace(-1, 1, res)
print(bins)



points_res_scaled = (points_array + 1) / 2 * (res-1)
rounded_array = np.round(points_res_scaled)
points_res = rounded_array / (res-1) * 2 - 1





print(rounded_array)


print(points_res)


print(np.round(points_array * (res-1) / 2) / (res-1)*2)

[-1.  -0.6 -0.2  0.2  0.6  1. ]
[[3. 0. 3. 1. 5. 1.]
 [3. 0. 2. 1. 5. 4.]]
[[ 0.2 -1.   0.2 -0.6  1.  -0.6]
 [ 0.2 -1.  -0.2 -0.6  1.   0.6]]
[[ 0.4 -0.8  0.4 -0.4  0.8 -0.8]
 [ 0.4 -0.8  0.  -0.4  0.8  0.8]]


In [ ]:
import numpy as np

points_array = [[0.242, -1, 0.312421, -0.4124, 1],
                [0.254, -1, 0., -0.442142, 1]
                ]

# Assuming your array is named 'points_array'
# Specify the number of bins
num_bins = 5


# Define the bins
bins = np.linspace(-1, 1, num_bins)

print(bins)
# Digitize the points
bin_indices = np.digitize(points_array, bins[:])

print(bin_indices)

# Replace the original values with the bin values



#binned_values = np.array([bins[i - 1    ] if i > 0 and i < len(bins) else np.nan for i in bin_indices])

binned_values = bins[bin_indices-1]


print(binned_values)

[-1.  -0.5  0.   0.5  1. ]
[[3 1 3 2 5]
 [3 1 3 2 5]]
[[ 0.  -1.   0.  -0.5  1. ]
 [ 0.  -1.   0.  -0.5  1. ]]


In [53]:
def round_to_nearest_numpy(x, nearest_lower, nearest_higher):
    return np.where(abs(x - nearest_lower) < abs(x - nearest_higher), nearest_lower, nearest_higher)

In [57]:
pints = round_to_nearest_numpy(points_array, bins[bin_indices-1], bins[bin_indices])

IndexError: index 5 is out of bounds for axis 0 with size 5

In [ ]:
def round_to_nearest_multiple(x, bins0):
    differences = [abs(x - val) for val in bins0]
    
    nearest_index = np.argmin(differences)
    return bins0[nearest_index]

In [56]:
round_to_nearest_multiple(points_array, bins)

np.float64(-0.5)

In [2]:
from ProcessOptimizer.doe import get_optimal_DOE
from ProcessOptimizer.space import Integer, Real, Space

You give generator a Space object generated by the `ProcessOptimizer` Space class.

ALSO INFORMATION ABOUT HTE OTHER PARAMETERS

In [4]:
factor_space = Space(dimensions=[Real(10, 40, name='ul_indicator'),
                                     Integer(20, 100, name='ul_base'),
                                     Integer(20, 100, name='ul_acid'),
                                     ])

design, factor_names = get_optimal_DOE(factor_space, 10, design_type='response')

In [5]:
print(design)
print(factor_names)

[[20.90909090909091, 100, 100], [10.0, 20, 100], [10.0, 20, 20], [40.0, 100, 49], [29.09090909090909, 71, 20], [40.0, 42, 100], [40.0, 20, 20], [10.0, 100, 20], [26.363636363636363, 20, 64], [10.0, 71, 71]]
['ul_indicator', 'ul_base', 'ul_acid']


NOTE categorical variables are not yet supported.

In [6]:
factor_names = factor_space.names
my_model1 = f'{factor_names[0]} + {factor_names[1]} + {factor_names[2]} + {factor_names[0]}:{factor_names[1]} + {factor_names[0]}:{factor_names[2]} + {factor_names[1]}:{factor_names[2]}'
print(my_model1)

ul_indicator + ul_base + ul_acid + ul_indicator:ul_base + ul_indicator:ul_acid + ul_base:ul_acid


In [8]:
get_optimal_DOE(factor_space, 7, model=my_model1)

([[10.0, 100, 20],
  [10.0, 100, 100],
  [40.0, 20, 100],
  [40.0, 100, 100],
  [10.0, 20, 100],
  [40.0, 100, 20],
  [10.0, 20, 20]],
 ['ul_indicator', 'ul_base', 'ul_acid'])

# OLD BELOW

In [12]:
my_model2 = f'{factor_names[0]} + {factor_names[1]} + {factor_names[2]} + pow({factor_names[1]}, 2)'
print(my_model2)

ul_indicator + ul_base + ul_acid + pow(ul_base, 2)


In [1]:
get_optimal_DOE(factor_space, 3, model=my_model2)

NameError: name 'get_optimal_DOE' is not defined

In [ ]:
get_optimal_DOE(factor_space, 5, model=my_model2)